**Step 1: Setup Data**

*Upload the CSV File*

In [29]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore (1).csv


**Read the CSV**

In [30]:
import pandas as pd

df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

In [31]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Create a SQLite Database Connection**





Import the Superstore dataset into a table named superstore_raw.

In [37]:
import sqlite3

conn = sqlite3.connect("superstore.db")

In [33]:
df.to_sql("superstore_raw", conn, if_exists="replace", index=False)

9994

**Verify the Table Was Created**

In [34]:
query = "SELECT * FROM superstore_raw LIMIT 5"

pd.read_sql(query, conn)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


**Create customers Table**

Insert data into  tables using SELECT DISTINCT

In [44]:
conn.execute("""
CREATE TABLE IF NOT EXISTS customers AS
SELECT DISTINCT
    "Customer ID" AS customer_id,
    "Customer Name" AS customer_name,
    Segment,
    Country,
    City,
    State,
    Region
FROM superstore_raw;
""")

In [39]:
pd.read_sql("SELECT * FROM customers LIMIT 5", conn)

,customer_id,customer_name,Segment,Country,City,State,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,South


**Create products Table**


In [45]:
conn.execute("""
CREATE TABLE IF NOT EXISTS products AS
SELECT DISTINCT
    "Product ID" AS product_id,
    "Product Name" AS product_name,
    Category,
    "Sub-Category" AS sub_category
FROM superstore_raw;
""")

In [58]:
pd.read_sql("SELECT * FROM products LIMIT 5", conn)

,product_id,product_name,Category,sub_category
0,FUR-BO-10001798,Bush Somerset Collection Bookcase,Furniture,Bookcases
1,FUR-CH-10000454,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",Furniture,Chairs
2,OFF-LA-10000240,Self-Adhesive Address Labels for Typewriters b...,Office Supplies,Labels
3,FUR-TA-10000577,Bretford CR4500 Series Slim Rectangular Table,Furniture,Tables
4,OFF-ST-10000760,Eldon Fold 'N Roll Cart System,Office Supplies,Storage


**Create orders Table**

In [46]:
conn.execute("""
CREATE TABLE  IF NOT EXISTS orders AS
SELECT DISTINCT
    "Order ID" AS order_id,
    "Order Date" AS order_date,
    "Ship Date" AS ship_date,
    "Ship Mode" AS ship_mode,
    "Customer ID" AS customer_id,
    "Product ID" AS product_id,
    Sales,
    Quantity,
    Discount,
    Profit
FROM superstore_raw;
""")

In [47]:
pd.read_sql("SELECT * FROM orders LIMIT 5", conn)

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,OFF-LA-10000240,14.6200,2,0.00,6.8714
3,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,OFF-ST-10000760,22.3680,2,0.20,2.5164


**Check Record Counts**

In [48]:
print(pd.read_sql("SELECT COUNT(*) AS customers FROM customers", conn))

print(pd.read_sql("SELECT COUNT(*) AS products FROM products", conn))

print(pd.read_sql("SELECT COUNT(*) AS orders FROM orders", conn))

   customers
0       4688
   products
0      1894
   orders
0    9993


**Step 2: Perform Required Queries**

1. Find all orders where sales are greater than the average sales. (Subquery)

In [61]:
query = """
SELECT *
FROM orders
WHERE Sales > (
    SELECT AVG(Sales)
    FROM orders
);
"""
pd.read_sql(query, conn)

,order_id,order_date,ship_date,ship_mode,customer_id,product_id,Sales,Quantity,Discount,Profit
0,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-BO-10001798,261.9600,2,0.00,41.9136
1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,FUR-CH-10000454,731.9400,3,0.00,219.5820
2,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,FUR-TA-10000577,957.5775,5,0.45,-383.0310
3,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,TEC-PH-10002275,907.1520,6,0.20,90.7152
4,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,FUR-TA-10001539,1706.1840,9,0.20,85.3092
...,...,...,...,...,...,...,...,...,...,...
2354,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,TEC-PH-10004080,271.9600,5,0.20,27.1960
2355,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,TEC-PH-10002496,249.5840,2,0.20,31.1980
2356,US-2016-103674,12/6/2016,12/10/2016,Standard Class,AP-10720,OFF-BI-10002026,437.4720,14,0.20,153.1152
2357,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,TEC-PH-10003645,258.5760,2,0.20,19.3932


2. Find the highest sales order for each customer. (Subquery)

In [62]:
query = """
SELECT
    customer_id,
    order_id,
    Sales
FROM orders o
WHERE Sales = (
    SELECT MAX(Sales)
    FROM orders
    WHERE customer_id = o.customer_id
);
"""

pd.read_sql(query, conn)

,customer_id,order_id,Sales
0,CG-12520,CA-2016-152156,731.9400
1,SO-20335,US-2015-108966,957.5775
2,BH-11710,CA-2014-115812,1706.1840
3,EB-13870,CA-2015-106320,1044.6300
4,TB-21520,US-2015-150630,3083.4300
...,...,...,...
790,DH-13075,CA-2015-159534,1087.9360
791,IM-15055,CA-2016-129630,2799.9600
792,HW-14935,CA-2017-121559,2405.2000
793,RB-19435,CA-2017-153871,735.9800


3. Calculate total sales for each customer. (CTE)

In [63]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT *
FROM customer_sales
ORDER BY total_sales DESC;
"""

pd.read_sql(query, conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
788,RS-19870,22.328
789,MG-18205,16.739
790,CJ-11875,16.520
791,LD-16855,5.304


4. Find customers whose total sales are above average. (CTE + Subquery)

In [64]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT *
FROM customer_sales
WHERE total_sales > (
    SELECT AVG(total_sales)
    FROM customer_sales
)
ORDER BY total_sales DESC;
"""

pd.read_sql(query, conn)

,customer_id,total_sales
0,SM-20320,25043.050
1,TC-20980,19052.218
2,RB-19360,15117.339
3,TA-21385,14595.620
4,AB-10105,14473.571
...,...,...
289,JK-16120,2932.484
290,SW-20455,2921.544
291,ML-17410,2921.500
292,RD-19585,2912.894


5. Rank all customers based on total sales. (Window Function)

In [65]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    customer_id,
    total_sales,
    RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
FROM customer_sales;
"""

pd.read_sql(query, conn)

,customer_id,total_sales,sales_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


6. Assign row numbers to each order within a customer. (Window Function + PARTITION BY)

In [66]:
query = """
SELECT
    customer_id,
    order_id,
    Sales,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY Sales DESC
    ) AS row_num
FROM orders;
"""

pd.read_sql(query, conn)

,customer_id,order_id,Sales,row_num
0,AA-10315,CA-2016-103982,3930.072,1
1,AA-10315,CA-2014-128055,673.568,2
2,AA-10315,CA-2016-103982,431.976,3
3,AA-10315,CA-2017-147039,362.940,4
4,AA-10315,CA-2014-128055,52.980,5
...,...,...,...,...
9988,ZD-21925,CA-2017-141481,61.440,5
9989,ZD-21925,CA-2014-143336,22.720,6
9990,ZD-21925,US-2016-147991,16.720,7
9991,ZD-21925,CA-2016-152471,15.984,8


7. Display top 3 customers based on total sales. (Window Function)

In [67]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
),
ranked_customers AS (
    SELECT
        customer_id,
        total_sales,
        RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
    FROM customer_sales
)
SELECT *
FROM ranked_customers
WHERE sales_rank <= 3;
"""

pd.read_sql(query, conn)

,customer_id,total_sales,sales_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


**Step 3: Final Combined Query**

Write one final query that shows:

  Customer Name  

  Total Sales  

  Rank  

(Use JOIN + CTE + Window Function together)

In [68]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    cs.total_sales,
    RANK() OVER (ORDER BY cs.total_sales DESC) AS customer_rank
FROM customer_sales cs
JOIN customers c
    ON cs.customer_id = c.customer_id
ORDER BY customer_rank;
"""

pd.read_sql(query, conn)

,customer_name,total_sales,customer_rank
0,Sean Miller,25043.050,1
1,Sean Miller,25043.050,1
2,Sean Miller,25043.050,1
3,Sean Miller,25043.050,1
4,Sean Miller,25043.050,1
...,...,...,...
4683,Mitch Gastineau,16.739,4684
4684,Carl Jackson,16.520,4685
4685,Lela Donovan,5.304,4686
4686,Thais Sissman,4.833,4687


**Mini Project: Customer Sales Insights**

Answer the following using SQL:

Who are the top 5 customers?  

Who are the bottom 5 customers?  

Which customers made only one order?  

Which customers have above-average sales?  

What is the highest order value per customer?

1. Top 5 Customers

In [69]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    cs.total_sales
FROM customer_sales cs
JOIN customers c
    ON cs.customer_id = c.customer_id
ORDER BY cs.total_sales DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,customer_name,total_sales
0,Sean Miller,25043.05
1,Sean Miller,25043.05
2,Sean Miller,25043.05
3,Sean Miller,25043.05
4,Sean Miller,25043.05


2. Bottom 5 Customers

In [70]:
query = """
WITH customer_sales AS (
    SELECT
        customer_id,
        SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    cs.total_sales
FROM customer_sales cs
JOIN customers c
    ON cs.customer_id = c.customer_id
ORDER BY cs.total_sales ASC
LIMIT 5;
"""

pd.read_sql(query, conn)

,customer_name,total_sales
0,Thais Sissman,4.833
1,Thais Sissman,4.833
2,Lela Donovan,5.304
3,Carl Jackson,16.520
4,Mitch Gastineau,16.739


3. Customers Who Made Only One Order

In [71]:
query = """
SELECT
    c.customer_name,
    COUNT(o.order_id) AS order_count
FROM orders o JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id, c.customer_name
HAVING COUNT(o.order_id) = 1;
"""

pd.read_sql(query, conn)

,customer_name,order_count
0,Anthony O'Donnell,1
1,Carl Jackson,1
2,Jocasta Rupert,1
3,Lela Donovan,1
4,Ricardo Emerson,1


4. Customers with Above-Average Sales

In [72]:
query = """
WITH customer_sales AS (
    SELECT
customer_id,
SUM(Sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT
    c.customer_name,
    cs.total_sales
FROM customer_sales cs
JOIN customers c
    ON cs.customer_id = c.customer_id
WHERE cs.total_sales > (
    SELECT AVG(total_sales)
    FROM customer_sales
)
ORDER BY cs.total_sales DESC;
"""

pd.read_sql(query, conn)

,customer_name,total_sales
0,Sean Miller,25043.050
1,Sean Miller,25043.050
2,Sean Miller,25043.050
3,Sean Miller,25043.050
4,Sean Miller,25043.050
...,...,...
2106,Craig Yedwab,2900.026
2107,Craig Yedwab,2900.026
2108,Craig Yedwab,2900.026
2109,Craig Yedwab,2900.026


5. Highest Order Value per Customer

In [73]:
query = """
SELECT
    c.customer_name,
    o.order_id,
    o.Sales AS highest_order_value
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.Sales = ( SELECT MAX(Sales) FROM orders  WHERE customer_id = o.customer_id
)
ORDER BY c.customer_name;
"""

pd.read_sql(query, conn)

,customer_name,order_id,highest_order_value
0,Aaron Bergman,CA-2016-140935,341.96
1,Aaron Bergman,CA-2016-140935,341.96
2,Aaron Bergman,CA-2016-140935,341.96
3,Aaron Hawkins,CA-2015-130113,668.16
4,Aaron Hawkins,CA-2015-130113,668.16
...,...,...,...
4697,Zuschuss Donatelli,CA-2016-152471,823.96
4698,Zuschuss Donatelli,CA-2016-152471,823.96
4699,Zuschuss Donatelli,CA-2016-152471,823.96
4700,Zuschuss Donatelli,CA-2016-152471,823.96
